In [ ]:
!pip install bitsandbytes trl

In [ ]:
# 깃허브에서 compatibility_functions.py 파일을 다운로드합니다.
!wget https://raw.githubusercontent.com/rickiepark/fine-tuning-llm/refs/heads/main/compatibility_functions.py

In [ ]:
!wget https://raw.githubusercontent.com/rickiepark/fine-tuning-llm/refs/heads/main/helper_functions.py

In [4]:
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset,Dataset
from peft import get_peft_model, prepare_model_for_kbit_training, LoraConfig, AutoPeftModelForCausalLM
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from helper_functions import *
from compatibility_functions import DataCollatorForCompletionOnlyLM

In [5]:
supported=torch.cuda.is_bf16_supported(including_emulation=False)
compute_dtype=(torch.bfloat16 if supported else torch.float32)

nf4_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

model_q4=AutoModelForCausalLM.from_pretrained(
    "facebook/opt-350m",device_map="cuda:0",quantization_config=nf4_config
)

model_q4=prepare_model_for_kbit_training(model_q4)

config=LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
peft_model=get_peft_model(model_q4,config)

tokenizer=AutoTokenizer.from_pretrained("facebook/opt-350m")
tokenizer=modify_tokenizer(tokenizer)
tokenizer=add_template(tokenizer)

peft_model=modify_model(peft_model,tokenizer)

dataset=load_dataset("dvgodoy/yoda_sentences",split="train")
dataset=dataset.rename_column("sentence","prompt")
dataset=dataset.rename_column("translation_extra","completion")

dataset=dataset.map(format_dataset)
dataset=dataset.remove_columns(["prompt","completion","translation"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/662M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentences.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/720 [00:00<?, ? examples/s]

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

In [6]:
mvt_trainer=SFTTrainer(
    model=peft_model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./future_model_name_on_the_hub",
        report_to="none"
    )
)

Tokenizing train dataset:   0%|          | 0/720 [00:00<?, ? examples/s]

In [7]:
mvt_train_dataloader=mvt_trainer.get_train_dataloader()
batch=next(iter(mvt_train_dataloader))
batch["input_ids"][:2]

tensor([[50265, 12105, 50118,  1213,   362,    49,  1159,    31,     5,   285,
           334,     4,     2, 50118, 50265,  2401, 33388, 50118,  7605,     5,
           285,   334,     6,    49,  1159,    51,   362,     4,   289,   338,
         41311,     4,     2, 50118,     1,     1,     1,     1,     1,     1,
             1,     1],
        [50265, 12105, 50118,   250,  9371,    16,    99, 33199,   460,  1733,
            11,     4,     2, 50118, 50265,  2401, 33388, 50118,  2264, 33199,
           460,  1733,    11,     6,    10,  9371,    16,     4,   854, 47820,
          7485,    29,     4,     2, 50118,     1,     1,     1,     1,     1,
             1,     1]], device='cuda:0')

In [8]:
tokenizer.pad_token_id,tokenizer.padding_side

(1, 'right')

In [9]:
pack_trainer=SFTTrainer(
    model=peft_model,
    processing_class=tokenizer,
    train_dataset=dataset,
    data_collator=None,
    args=SFTConfig(output_dir="./future_name_on_the_hub",
                   packing=True,
                   packing_strategy="wrapped",
                   max_length=64,
                   report_to="none",
                   bf16=supported)
)

Packing train dataset:   0%|          | 0/720 [00:00<?, ? examples/s]

In [10]:
pack_train_dataloader=pack_trainer.get_train_dataloader()
batch=next(iter(pack_train_dataloader))
batch["input_ids"][:2]

tensor([[50265, 12105, 50118,   133,  4716,  1536,  1136,    19,     5,   220,
         32322,     9,  2508,     4,     2, 50118, 50265,  2401, 33388, 50118,
          3908,     5,   220, 32322,     9,  2508,     6,     5,  4716,  1536,
          1136,     4,  3216,     6,  1368, 28015, 41311,     4,     2, 50118,
         50265, 12105, 50118, 34306,   110,   275, 35310,     7,     5,   371,
          1380,     4,     2, 50118, 50265,  2401, 33388, 50118,  3972,     5,
           371,  1380,     6,   836],
        [12105, 50118, 29729,   661,     5,  7511,     7,   110,   314,  4793,
             4,     2, 50118, 50265,  2401, 33388, 50118, 29729,   661,     5,
          7511,     7,   110,   314,  4793,     6,    47,   531,     4,     2,
         50118, 50265, 12105, 50118, 21031,     5, 20058,  2718,     7,  1338,
             5,  8037,     4,     2, 50118, 50265,  2401, 33388, 50118,  3972,
          1338,     5,  8037,     6,   185,     5, 20058,  2718,     6,    47,
           531

In [11]:
tokenizer.padding_side="left"

response_template="<|im_start|>assistant\n"
collator_fn=DataCollatorForCompletionOnlyLM(response_template,tokenizer=tokenizer)
completions_trainer=SFTTrainer(
    model=peft_model,
    processing_class=tokenizer,
    train_dataset=dataset,
    data_collator=collator_fn,
    args=SFTConfig(output_dir="./future_name_on_the_hub",
                   packing=False,
                   max_length=64,
                   report_to="none",
                   bf16=supported)
)

Tokenizing train dataset:   0%|          | 0/720 [00:00<?, ? examples/s]

In [12]:
completions_train_dataloader= completions_trainer.get_train_dataloader()
batch=next(iter(completions_train_dataloader))
input_ids=batch["input_ids"][0]
labels=batch["labels"][0]
input_ids,labels

(tensor([    1,     1,     1,     1,     1,     1,     1,     1, 50265, 12105,
         50118,  1213,   362,    49,  1159,    31,     5,   285,   334,     4,
             2, 50118, 50265,  2401, 33388, 50118,  7605,     5,   285,   334,
             6,    49,  1159,    51,   362,     4,   289,   338, 41311,     4,
             2, 50118], device='cuda:0'),
 tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  7605,     5,   285,   334,
             6,    49,  1159,    51,   362,     4,   289,   338, 41311,     4,
             2, 50118], device='cuda:0'))

In [13]:
print(tokenizer.decode(input_ids))

<pad><pad><pad><pad><pad><pad><pad><pad><|im_start|>user
They took their kids from the public school.</s>
<|im_start|>assistant
From the public school, their kids they took. Hrmmm.</s>



In [14]:
valid=labels>=0
print(tokenizer.decode(labels[valid]))

From the public school, their kids they took. Hrmmm.</s>



In [15]:
trainer=SFTTrainer(
    model=peft_model,
    processing_class=tokenizer,
    train_dataset=dataset,
    data_collator=None,
    args=SFTConfig(output_dir="./future_name_on_the_hub",
                   packing=True,
                   packing_strategy="wrapped",
                   max_length=64,
                   report_to="none",
                   bf16=supported)
)

Packing train dataset:   0%|          | 0/720 [00:00<?, ? examples/s]

In [16]:
trainer.train()

Step,Training Loss
10,5.290432
20,4.979301
30,4.656644
40,4.391819
50,4.195033
60,4.000849
70,3.893972
80,3.754433
90,3.666494
100,3.634140


TrainOutput(global_step=144, training_loss=4.035546488232082, metrics={'train_runtime': 74.2911, 'train_samples_per_second': 15.507, 'train_steps_per_second': 1.938, 'total_flos': 134891635212288.0, 'train_loss': 4.035546488232082, 'entropy': 3.6989776492118835, 'num_tokens': 73695.0, 'mean_token_accuracy': 0.4122023805975914, 'epoch': 3.0})

In [17]:
print(generate(trainer.model,tokenizer,"There is bacon in this sandwich."))

<|im_start|>user
There is bacon in this sandwich.</s>
<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant

<|im_start|>assistant


In [18]:
(trainer.args.learning_rate,
 trainer.args.per_device_train_batch_size,
 trainer.args.num_train_epochs)

(2e-05, 8, 3.0)

In [19]:
trainer.optimizer.__class__

accelerate.optimizer.AcceleratedOptimizer

In [20]:
trainer.optimizer.optimizer.__class__

torch.optim.adamw.AdamW

In [21]:
#train() method를 호출했을 때 일어나는 모든 작업을 단순화하여 압축시킴
n_epochs=trainer.args.num_train_epochs
#packing과 collator 설정에 따라 data loader을 가져온다.
train_dataloader=trainer.get_train_dataloader()
#단계 0
trainer.create_optimizer()
#args에 전달된 모든 객체를 분산 훈련과 혼합 정밀도를 위해 준비하고 동일한 순서대로 반환한다.
trainer.model, trainer.optimizer=trainer.accelerator.prepare(
    trainer.model,trainer.optimizer
)
#필요하면 gradient checkpointing을 활성화한다.
if trainer.args.gradient_checkpointing:
  trainer.model.gradient_checkpointing_enable(
      gradient_checkpointing_kwargs={"use_reentrant":False}
  )
trainer.model.zero_grad()

for epoch in range(int(n_epochs)):
  for step, inputs in enumerate(train_dataloader):
    #gradient 누적을 준비한다.
    with trainer.accelerator.accumulate(trainer.model):
      trainer.model.train()

      #단계1
      #재귀적으로 list/tuple/dictionary를 순회하면서
      #모든 tensor를 trainer.args.device에 전송한다.
      inputs=trainer._prepare_inputs(inputs)
      #forward pass
      outputs=trainer.model(**inputs)
      #micro batch loss을 추출한다.
      loss=(outputs["loss"] if isinstance(outputs,dict) else outputs[0])

      #단계2
      #backward pass - backpropagation
      trainer.accelerator.backward(loss)

      #단계3
      #parameter update
      trainer.optimizer.step()

      #단계4
      #gradient 재설정
      trainer.model.zero_grad()

    supported=torch.cuda.is_bf16_supported(including_emulation=False)
    trainer=SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=peft_config,
        args=SFTConfig(output_dir="./future_name_on_the_hub",
                      packing=True,
                      packing_strategy="wrapped",
                      max_length=max_seq_length,
                      report_to="none",
                      bf16=supported)
    )

    supported=torch.cuda.isbf16_supported(including_emulation=False)
    tokenzier.padding_side="left"
    response_template="..."
    collator_fn=DataCollatorForCompletionOnlyLM(
        response_template,tokenizer=tokenizer
    )
    trainer=SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=peft_config,
        data_collator=collator_fn,
        args=SFTConfig(output_dir="./future_name_on_the_hub",
                      packing=False,
                      max_length=max_seq_length,
                      report_to="none")
    )

In [22]:
dataloader=trainer.get_train_dataloader()
batch=next(iter(dataloader))
labels=batch["labels"][0]
valid=labels>=0
print(tokenizer.decode(labels[valid]))

<|im_start|>user
The petals fall with the next puff of wind.</s>
<|im_start|>assistant
With the next puff of wind, the petals fall. Yes, hrrmmm.</s>
<|im_start|>user
Bring your best compass to the third class.</s>
<|im_start|>assistant
To the third class, bring


In [23]:
#batch size 자동 찾기
def find_max_batch_size(trainer,initial_batch_size=None,update_trainer=False):
  from copy import deepcopy
  from accelerate.utils import find_executable_batch_size
  #동일 model을 사용하여 새로운 dummy Trainer object를 만든다.
  new_trainer=SFTTrainer(model=trainer.model)
  #저수준에서 전체 설정을 복제한다.
  new_trainer.__dict__.update(**trainer.__dict__)
  #SFTConfig을 수정하기 때문에 deepcopy로 복사한다.
  new_trainer.args=deepcopy(new_trainer.args)
  #학습률을 0으로 설정하므로 모델이 업데이트 안된다.
  new_trainer.args.learning_rate=0
  #OOM만 확인하면 되므로 1step만 실행한다.
  new_trainer.args.max_steps=1

  #이 함수를 사용한다면 실제 Trainer에 이를 지정할 필요가 없다.
  new_trainer.args.auto_find_batch_size=True
  #시작점으로 원본 'per_device_train_batch_size'를 사용하지만, 바꿀 수 있다.
  if initial_batch_size is not None:
    new_trainer.args.per_device_train_batch_size=initial_batch_size

  #내부적으로 일어나는 일은 다음과 같다.
  #내부 훈련 루프를 기바능로 decorate된 함수를 만든다.
  func=find_executable_batch_size(new_trainer._inner_training_loop)
  #그 다음 복사된 매개변수로 훈련 루프를 실행한다.
  #예외가 일어날 때마다 batch size를 절반으로 줄인다.
  #성공할 때까지 이를 반복하여 시도한다.
  #lr=0이므로 모델이 변경되지 않는다.
  func(args=new_trainer.args)
  #'_train_batch_size'속성에 성공한 batch size가 있다.
  max_batch_size=new_trainer._train_batch_size
  del new_trainer
  #편의상 batch size를 직접 업데이트할 수 있다.
  if update_trainer:
    trainer.args.per_device_train_batch_size=max_batch_size

  return max_batch_size

In [24]:
def peft_module_casting_to_bf16(model):
  for name, module in model.named_modules():
    if isinstance(module, torch.nn.LayerNorm) or "norm" in name:
      module=module.to(torch.float32)
    elif any(x in name for x in ["lm_head","embed_tokens","wte","wpe"]):
      if hasattr(module,"weight"):
        if module.weight.dtype==torch.float32:
            module=module.to(torch.bfloat16)

#실제 훈련

In [25]:
supported = torch.cuda.is_bf16_supported(including_emulation=False)
min_effective_batch_size=8
lr=3e-4
max_seq_length=64
collator_fn=None
packing=(collator_fn is None)
steps=20
num_train_epochs=10

sft_config = SFTConfig(
    output_dir='./future_name_on_the_hub',
    #dataset
    packing=packing,
    packing_strategy='wrapped',
    max_length=max_seq_length,
    #gradient/memory
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant':False},
    gradient_accumulation_steps=2,
    per_device_train_batch_size=min_effective_batch_size,
    auto_find_batch_size=True,
    #training
    num_train_epochs=num_train_epochs,
    learning_rate=lr,
    #config and logging
    report_to="tensorboard",
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=steps,
    save_strategy="steps",
    save_steps=steps,
    bf16=supported
)

trainer=SFTTrainer(
    model=peft_model,
    processing_class=tokenizer,
    train_dataset=dataset,
    data_collator=collator_fn,
    args=sft_config
)
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
20,2.713185
40,2.150626
60,1.943822
80,1.843059
100,1.784842
120,1.741521
140,1.713886
160,1.677742
180,1.645216
200,1.661949


TrainOutput(global_step=240, training_loss=1.8462685426076253, metrics={'train_runtime': 204.4418, 'train_samples_per_second': 18.783, 'train_steps_per_second': 1.174, 'total_flos': 449638784040960.0, 'train_loss': 1.8462685426076253, 'epoch': 10.0})

In [26]:
#model이 yoda처럼 말하는가
print(generate(trainer.model,tokenizer,"There is bacon in this sandwich."))

<|im_start|>user
There is bacon in this sandwich.</s>
<|im_start|>assistant
In this sandwich, bacon is. Yes, hrrrm.</s>


In [27]:
print(generate(trainer.model,tokenizer,"Then why are you crying?"))

<|im_start|>user
Then why are you crying?</s>
<|im_start|>assistant
Crying, you are, but why are you?</s>


In [28]:
print(generate(trainer.model,tokenizer,"Will you still love me when I'm no longer young and beautiful?"))

<|im_start|>user
Will you still love me when I'm no longer young and beautiful?</s>
<|im_start|>assistant
Will you still love me when I'm no longer young and beautiful? Yes, hrrrm.</s>


In [29]:
#adapter 저장
trainer.save_model("yoda-adapter")

In [30]:
!pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 103.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [31]:
#adapter config을 기반으로 base model을 자동으로 로드하고 그 위에 adapter를 로드한다.
reloaded_model=AutoPeftModelForCausalLM.from_pretrained("yoda-adapter")
reloaded_model

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=16, bias=False

In [32]:
merged_model=reloaded_model.merge_and_unload()
merged_model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=409

#Attention

In [33]:
#attention 공식에서 Q와 K가 정규 분포라면 d_k의 제곱근은 Q와 K의 dot product의 표준편차와 거의 같다.
import math
import torch

d_k=1024
q_vector=torch.randn(1000,1,d_k)
v_vector=torch.randn(1000,1,d_k).permute(0,2,1)
torch.bmm(q_vector,v_vector).squeeze().std(), math.sqrt(d_k)

(tensor(32.8390), 32.0)

In [34]:
#muti-head attention
class MultiHeadedAttention(nn.Module):
  def __init__(self, n_heads, d_model,dropout=0.1,verbose=False):
    super(MultiHeadedAttention,self).__init__()
    self.n_heads=n_heads
    self.d_model=d_model
    self.d_k=int(d_model/n_heads)
    self.linear_query=nn.Linear(d_model,d_model)
    self.linear_key=nn.Linear(d_model,d_model)
    self.linear_value=nn.Linear(d_model,d_model)
    self.linear_out=nn.Linear(d_model,d_model)
    self.dropout=nn.Dropout(p=dropout)
    self.alphas=None
    self.verbose=verbose

  def print_sizes(self,name,tensor):
    if self.verbose:
      print(f"{name:<22} - 크기: {str(tensor.shape):<30}"
            f"- 원소 개수: {str(torch.numel(tensor)):>10}")

  def make_chunks(self,x):
    batch_size, seq_len=x.size(0),x.size(1)
    #N,L,D -> N,L,n_heads*d_k
    x=x.view(batch_size,seq_len,self.n_heads,self.d_k)
    #N, n_heads, L, d_k
    x=x.transpose(1,2)
    return x

  def init_keys(self,key):
    #N, n_heads, L, d_k
    self.proj_key=self.make_chunks(self.linear_key(key))
    self.proj_value=self.make_chunks(self.linear_value(key))

  def alignment_function(self,query):
    #scaled dot product attention
    #N, n_heads, L, d_k x # N, n_heads, d_k, L -> N, n_heads, L, L
    proj_query=self.make_chunks(self.linear_query(query))
    dot_products=torch.matmul(proj_query,self.proj_key.transpose(-2,-1))
    scores=dot_products/np.sqrt(self.d_k)
    self.print_sizes("쿼리 투영",proj_query)
    self.print_sizes("키 투영",self.proj_key)
    self.print_sizes("값 투영",self.proj_value)
    self.print_sizes("점곱",dot_products)
    self.print_sizes("정렬",scores)
    return scores
  def attn(self, query, mask=None):
    #query는 batch dimension이 먼저 등장한다: N, L, D
    #정렬 함수는 각 head에 대한 정렬 결과를 계산한다.
    #N, n_heads, L, L
    alignments = self.alignment_function(query)
    if mask is not None:
      alignments = alignments.masked_fill(mask == 0, -1e9)
    alphas=F.softmax(alignments,dim=-1) #N, n_heads, L, L
    alphas=self.dropout(alphas)
    self.alphas=alphas.detach()
    #N, n_heads, L, L x N, n_heads, L, d_k -> N, n_heads, L, d_k
    context=torch.matmul(alphas, self.proj_value)
    self.print_sizes("어텐션 가중치 / 알파",alphas)
    self.print_sizes("문맥 벡터 (헤드)",context)
    return context

  def output_function(self, contexts):
    #N, L, D
    out=self.linear_out(contexts) #N, L, D
    self.print_sizes("출력 문맥 벡터",out)
    return out

  def forward(self, query, mask=None):
    self.init_keys(query)

    if mask is not None:
      #N, 1, L, L - 모든 head가 동일한 mask을 사용한다.
      mask=mask.unsqueeze(1)

    #N, n_heads, L, d_k
    context=self.attn(query,mask=mask)
    #N, L, n_heads, d_k
    context=context.transpose(1,2).contiguous()
    #N, L, n_heads * d_k = N, L, d_model
    context=context.view(query.size(0), -1, self.d_model)
    #N, L, d_model
    out=self.output_function(context)
    return out

In [35]:
#매개변수 설정

bsize=1         #N
seqlen=256      #L
d_model=1024    #D
n_heads=16      #head 개수
#d_k = D/n_heads = 64
input_batch = torch.randn(bsize,seqlen,d_model)

In [36]:
#multi-head attention mechanism의 instanse를 만들고 dummy batch를 전달한다.
mha=MultiHeadedAttention(n_heads=n_heads, d_model=d_model, verbose=True)
out=mha(input_batch)

쿼리 투영                  - 크기: torch.Size([1, 16, 256, 64])  - 원소 개수:     262144
키 투영                   - 크기: torch.Size([1, 16, 256, 64])  - 원소 개수:     262144
값 투영                   - 크기: torch.Size([1, 16, 256, 64])  - 원소 개수:     262144
점곱                     - 크기: torch.Size([1, 16, 256, 256]) - 원소 개수:    1048576
정렬                     - 크기: torch.Size([1, 16, 256, 256]) - 원소 개수:    1048576
어텐션 가중치 / 알파           - 크기: torch.Size([1, 16, 256, 256]) - 원소 개수:    1048576
문맥 벡터 (헤드)             - 크기: torch.Size([1, 16, 256, 64])  - 원소 개수:     262144
출력 문맥 벡터               - 크기: torch.Size([1, 256, 1024])    - 원소 개수:     262144
